# `kitai.query_translation` — User Guide

`kitai/query_translation.py` provides three query-transformation strategies that
expand a user question into a richer set of queries before retrieval.
Each strategy targets a different retrieval failure mode:

```
kitai/query_translation.py
│
├── Strategy functions          (require a LangChain chat model)
│   ├── decompose_query()       break complex question → focused sub-questions
│   ├── step_back_query()       specific question → broader abstract questions
│   └── expand_query()          question → paraphrased / synonym variants
│
└── Helper utilities            (no model required)
    ├── format_few_shot_examples()  format few-shot dicts → prompt string
    └── create_input_objects()      list[str] → list[dict] for chain.batch()
```

| Strategy | Output type | Why it helps |
|---|---|---|
| `decompose_query` | `list[DecomposedQuery]` | Multi-part questions often miss documents that only answer one part |
| `step_back_query` | `list[GeneralQuery]` | Retrieves foundational context that specific queries miss |
| `expand_query` | `list[ParaphrasedQuery]` | Synonyms and alternative phrasings widen recall |

**Prerequisites:**
- Sections 1–3 (helpers) run with **no API key**.
- Sections 4–6 (strategy functions) require an `OPENAI_API_KEY` in your `.env`.

**Related modules:**
- `kitai.index` — vector store and retrieval.
- `kitai.batch` — in-memory Batch API helpers for embedding at scale.

## Sections
1. [Setup](#1-setup)
2. [Helper: `format_few_shot_examples`](#2-helper-format_few_shot_examples)
3. [Helper: `create_input_objects`](#3-helper-create_input_objects)
4. [`decompose_query`](#4-decompose_query)
5. [`step_back_query`](#5-step_back_query)
6. [`expand_query`](#6-expand_query)
7. [Custom few-shot examples](#7-custom-few-shot-examples)
8. [End-to-end pattern](#8-end-to-end-pattern)
9. [Error handling reference](#9-error-handling-reference)

---
## 1 — Setup

In [ ]:
import logging
import os

from kitai.query_translation import (
    # strategy functions
    decompose_query,
    step_back_query,
    expand_query,
    # helpers
    format_few_shot_examples,
    create_input_objects,
    # default few-shot constants (importable for inspection / override)
    _DEFAULT_DECOMPOSE_EXAMPLES,
    _DEFAULT_STEP_BACK_EXAMPLES,
    _DEFAULT_EXPAND_EXAMPLES,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)

### Sample queries used throughout this guide

All strategy-function examples share these three questions.
The first two are intentionally complex or specific to illustrate
how each strategy transforms them.

In [ ]:
QUERIES = [
    "What are the coverage limits and exclusions for natural disasters in this policy?",
    "How does the indemnity clause interact with the subrogation rights?",
    "Who are the named insureds?",
]

print(f"Sample queries ({len(QUERIES)}):")
for i, q in enumerate(QUERIES, 1):
    print(f"  {i}. {q}")

---
## 2 — Helper: `format_few_shot_examples`

Converts a list of example dicts into the plain-text block injected
into each strategy's prompt.  Each dict must have two keys:

| Key | Type | Description |
|---|---|---|
| `original_query` | `str` | The seed question shown in the example |
| `new_queries` | `list[str]` | The generated queries for that seed |

The `label` parameter changes the section heading to match the
calling strategy (e.g. `"Sub-Queries"`, `"Step-Back Queries"`,
`"Paraphrased Queries"`).

**Invariant:** this function is pure — no model call, no side effects.

In [ ]:
# Inspect the built-in decompose examples as they appear inside the prompt
decompose_few_shot = format_few_shot_examples(
    _DEFAULT_DECOMPOSE_EXAMPLES,
    label="Sub-Queries",
)
print(decompose_few_shot)

In [ ]:
# Step-back label
step_back_few_shot = format_few_shot_examples(
    _DEFAULT_STEP_BACK_EXAMPLES,
    label="Step-Back Queries",
)
print(step_back_few_shot)

In [ ]:
# When there is exactly one example the header reads "Example:" not "Examples:"
single_example = _DEFAULT_DECOMPOSE_EXAMPLES[:1]
print(format_few_shot_examples(single_example, label="Sub-Queries"))

---
## 3 — Helper: `create_input_objects`

Converts a plain `list[str]` into the `list[dict]` format consumed
by `chain.batch()`.  Each string becomes `{"question": str}`.
Extra keyword arguments are merged into every dict unchanged — this is
how `few_shot_str` is broadcast to all batch items without repeating it.

**Why this exists:** LangChain's `PromptTemplate` requires named dicts,
not bare strings, as batch inputs.  This function is the single
conversion point so callers never manually build those dicts.

In [ ]:
# Basic — just the question field
basic = create_input_objects(["What is a deductible?", "What is a premium?"])
print("Basic output:")
for item in basic:
    print(f"  {item}")

In [ ]:
# With extra fields — few_shot_str is constant for all items
few_shot_str = format_few_shot_examples(_DEFAULT_EXPAND_EXAMPLES, label="Paraphrased Queries")

batch_inputs = create_input_objects(
    ["What is a deductible?", "What is a premium?"],
    few_shot_str=few_shot_str,
)

print(f"Items: {len(batch_inputs)}")
for item in batch_inputs:
    # truncate the few_shot_str for readability
    display = {k: (v[:60] + "...") if k == "few_shot_str" else v for k, v in item.items()}
    print(f"  {display}")

---
## 4 — `decompose_query`

Breaks a complex or multi-part question into a set of focused
sub-questions that can each be answered independently.

**When to use:** The original query spans several distinct concepts
(e.g. *coverage limits AND exclusions AND natural disasters*).
A single-vector search would need to compromise across all concepts;
decomposed sub-queries retrieve dedicated passages for each.

**Output shape:**
```
decompose_query(model, [q1, q2]) → [[DecomposedQuery, ...], [DecomposedQuery, ...]]
                                     ─── results for q1 ──   ─── results for q2 ───
```
Each inner list has as many `DecomposedQuery` objects as the model chose to generate.
Access the text via `.decomposed_query`.

> **Requires OPENAI_API_KEY.** The cells below are guarded so they
> won't run if the key is missing.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping decompose_query demo.")
else:
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    results = decompose_query(model, QUERIES)

    for original, sub_queries in zip(QUERIES, results):
        print(f"Original : {original}")
        print(f"Sub-queries ({len(sub_queries)}):")
        for sq in sub_queries:
            print(f"  - {sq.decomposed_query}")
        print()

### Flattening results for retrieval

Retrievers expect a flat `list[str]`.  Use a list comprehension to
flatten while preserving the source query for traceability.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping flatten demo.")
else:
    flat_sub_queries = [
        sq.decomposed_query
        for sub_list in results
        for sq in sub_list
    ]

    print(f"Total sub-queries: {len(flat_sub_queries)}")
    for q in flat_sub_queries:
        print(f"  - {q}")

---
## 5 — `step_back_query`

Transforms specific, narrow questions into broader conceptual questions
that expose the underlying principles — the "step-back prompting" technique.

**When to use:** The original query is too narrow to match passages that
contain the answer in a more general form (e.g., a specific clause question
whose answer lives in a general policy-interpretation passage).

**`num_queries` parameter:** Controls exactly how many step-back queries the
model generates per input.  This integer is baked into the prompt template
at call time; the LLM is instructed to produce exactly that many.

**Output shape:**
```
step_back_query(model, [q1, q2], num_queries=3)
    → [[GeneralQuery, GeneralQuery, GeneralQuery],   # for q1
       [GeneralQuery, GeneralQuery, GeneralQuery]]   # for q2
```
Access the text via `.general_query`.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping step_back_query demo.")
else:
    step_back_results = step_back_query(model, QUERIES, num_queries=3)

    for original, general_queries in zip(QUERIES, step_back_results):
        print(f"Original : {original}")
        print(f"Step-back queries ({len(general_queries)}):")
        for gq in general_queries:
            print(f"  - {gq.general_query}")
        print()

### Varying `num_queries`

For narrow, factual questions one or two step-back queries are usually
enough.  For broad analytical questions, five or more may help.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping.")
else:
    single_query = ["What is the renewal date in this agreement?"]

    for n in [1, 2, 5]:
        results_n = step_back_query(model, single_query, num_queries=n)
        print(f"num_queries={n}:")
        for gq in results_n[0]:
            print(f"  - {gq.general_query}")
        print()

---
## 6 — `expand_query`

Generates multiple semantically equivalent phrasings of the original
question — synonym substitution, structural re-ordering, acronym
expansion — to widen retrieval recall.

**When to use:** The corpus uses different terminology than the user
(e.g., user says "party" but the document says "insured" or "obligor").

**Note on unknowns:** The prompt instructs the model *not* to rephrase
acronyms or domain terms it does not recognise — they are passed through
unchanged to avoid hallucinated expansions.

**Output shape:**
```
expand_query(model, [q1, q2])
    → [[ParaphrasedQuery, ...],   # for q1
       [ParaphrasedQuery, ...]]   # for q2
```
Access the text via `.paraphrased_query`.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping expand_query demo.")
else:
    expand_results = expand_query(model, QUERIES)

    for original, paraphrases in zip(QUERIES, expand_results):
        print(f"Original : {original}")
        print(f"Paraphrases ({len(paraphrases)}):")
        for pq in paraphrases:
            print(f"  - {pq.paraphrased_query}")
        print()

---
## 7 — Custom few-shot examples

All three strategy functions accept an optional `few_shot_examples`
parameter that overrides the module-level defaults.  This is the primary
lever for domain adaptation — tailor the examples to your document corpus
and the model will produce more relevant queries.

**Schema** (same for all three strategies):
```python
few_shot_examples = [
    {
        "original_query": str,      # the seed question
        "new_queries":    list[str], # desired generated queries
    },
    ...
]
```

**Design advice:**
- One well-crafted example often outperforms three generic ones.
- The example should mirror the *type* of queries your users will ask.
- `new_queries` act as the target distribution — include the style,
  length, and specificity you want in the outputs.

In [ ]:
# Inspect the default examples before deciding whether to override
print("=== Default decompose examples ===")
for ex in _DEFAULT_DECOMPOSE_EXAMPLES:
    print(f"  original_query : {ex['original_query']}")
    print(f"  new_queries    : {ex['new_queries']}")
    print()

print("=== Default step-back examples ===")
for ex in _DEFAULT_STEP_BACK_EXAMPLES:
    print(f"  original_query : {ex['original_query']}")
    print(f"  new_queries    : {ex['new_queries']}")
    print()

print("=== Default expand examples ===")
for ex in _DEFAULT_EXPAND_EXAMPLES:
    print(f"  original_query : {ex['original_query']}")
    print(f"  new_queries    : {ex['new_queries']}")
    print()

In [ ]:
# Custom examples tailored to a medical-records corpus
medical_decompose_examples = [
    {
        "original_query": "What medications were prescribed and for how long?",
        "new_queries": [
            "Which medications appear in the prescription records?",
            "What is the prescribed dosage for each medication?",
            "What is the duration of each prescription?",
            "Are there any contraindications noted in the records?",
        ],
    },
]

# Preview what this will look like inside the prompt
preview = format_few_shot_examples(medical_decompose_examples, label="Sub-Queries")
print(preview)

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping custom few-shot demo.")
else:
    medical_query = ["What medications were prescribed and what were the side effects reported?"]

    # With default examples
    default_result = decompose_query(model, medical_query)
    print("Default few-shot:")
    for sq in default_result[0]:
        print(f"  - {sq.decomposed_query}")

    print()

    # With custom medical examples
    custom_result = decompose_query(
        model, medical_query, few_shot_examples=medical_decompose_examples
    )
    print("Custom few-shot:")
    for sq in custom_result[0]:
        print(f"  - {sq.decomposed_query}")

---
## 8 — End-to-end pattern

A typical retrieval pipeline augments the user question with one or more
translation strategies, de-duplicates the resulting query pool, then runs
all queries against the vector store and merges the retrieved chunks.

```
user_query
     │
     ├─ decompose_query()   →  sub-queries
     ├─ step_back_query()   →  abstract queries
     └─ expand_query()      →  paraphrase variants
             │
          flatten + deduplicate
             │
          vector store retrieval  (one call per augmented query)
             │
          merge + re-rank
             │
          LLM synthesis
```

The cell below demonstrates the aggregation step without a vector store
(replace `dummy_retrieve` with your real retriever).

In [ ]:
def aggregate_augmented_queries(
    model,
    user_questions: list[str],
    *,
    use_decompose: bool = True,
    use_step_back: bool = True,
    use_expand: bool = True,
    step_back_num: int = 2,
) -> dict[str, list[str]]:
    """
    Run selected strategies and return a dict mapping each original question
    to a de-duplicated flat list of augmented queries.

    The original question is always included in the output list (position 0)
    so the retriever always covers the literal user intent.
    """
    # Run selected strategies in a single batch each
    decomposed   = decompose_query(model, user_questions)   if use_decompose  else [[] for _ in user_questions]
    stepped_back = step_back_query(model, user_questions, num_queries=step_back_num) if use_step_back  else [[] for _ in user_questions]
    expanded     = expand_query(model, user_questions)      if use_expand     else [[] for _ in user_questions]

    output = {}
    for i, original in enumerate(user_questions):
        seen = set()
        all_queries = [original]  # original is always first
        seen.add(original)

        for sq in decomposed[i]:
            if sq.decomposed_query not in seen:
                all_queries.append(sq.decomposed_query)
                seen.add(sq.decomposed_query)

        for gq in stepped_back[i]:
            if gq.general_query not in seen:
                all_queries.append(gq.general_query)
                seen.add(gq.general_query)

        for pq in expanded[i]:
            if pq.paraphrased_query not in seen:
                all_queries.append(pq.paraphrased_query)
                seen.add(pq.paraphrased_query)

        output[original] = all_queries

    return output


if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping end-to-end demo.")
else:
    augmented = aggregate_augmented_queries(
        model,
        QUERIES[:1],           # run on the first query only to save tokens
        use_decompose=True,
        use_step_back=True,
        use_expand=True,
        step_back_num=2,
    )

    for original, queries in augmented.items():
        print(f"Original: {original}")
        print(f"Augmented pool ({len(queries)} queries):")
        for j, q in enumerate(queries):
            tag = "[original]" if j == 0 else ""
            print(f"  {j:2d}. {q} {tag}")

---
## 9 — Error handling reference

All public functions validate their inputs before any expensive work
(model call, prompt formatting).  Errors are raised immediately with
descriptive messages.

| Function | Guard | Error type |
|---|---|---|
| `create_input_objects` | non-list or empty `input_strings` | `ValueError` |
| `create_input_objects` | non-string element | `TypeError` |
| `decompose_query` | propagated from `create_input_objects` | `ValueError` / `TypeError` |
| `step_back_query` | propagated from `create_input_objects` | `ValueError` / `TypeError` |
| `expand_query` | propagated from `create_input_objects` | `ValueError` / `TypeError` |
| LLM / network errors | propagated from LangChain | varies (`LangChainError`, …) |

In [ ]:
# create_input_objects — wrong container type
try:
    create_input_objects("not a list")  # type: ignore
except ValueError as e:
    print(f"Non-list input:\n  {type(e).__name__}: {e}\n")

In [ ]:
# create_input_objects — empty list
try:
    create_input_objects([])
except ValueError as e:
    print(f"Empty list:\n  {type(e).__name__}: {e}\n")

In [ ]:
# create_input_objects — non-string element
try:
    create_input_objects(["valid query", 42])  # type: ignore
except TypeError as e:
    print(f"Non-string element:\n  {type(e).__name__}: {e}\n")

In [ ]:
# Strategy functions propagate the same guards
# (no model call is needed — the error is raised before the chain is invoked)
if not os.environ.get("OPENAI_API_KEY"):
    # Guards are input-level, so they fire regardless of API key presence
    _dummy_model = None  # model is never reached
else:
    _dummy_model = model

# We need a model object for the function signature, but the guards
# fire before the model is used — pass None to confirm.
for fn_name, fn, bad_input in [
    ("decompose_query",  decompose_query,  []),
    ("step_back_query",  step_back_query,  []),
    ("expand_query",     expand_query,     []),
]:
    try:
        fn(None, bad_input)  # type: ignore[arg-type]
    except ValueError as e:
        print(f"{fn_name}(empty list):\n  {type(e).__name__}: {e}\n")